# VLM-DENTAL - Phase 3: Supervised Fine-Tuning (SFT)

Run this notebook *after* you have successfully generated CoT traces using the Master Notebook. This notebook handles fine-tuning Qwen3.5-9B on those traces.

## 1. Setup Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Set this to True to save the repository code directly to Google Drive.
SAVE_CODE_TO_DRIVE = False

drive_path = "/content/drive/MyDrive/VLM-DENTAL"
colab_path = "/content/VLM-DENTAL"
work_dir = drive_path if SAVE_CODE_TO_DRIVE else colab_path

if SAVE_CODE_TO_DRIVE:
    %cd /content/drive/MyDrive
else:
    %cd /content

if not os.path.exists("VLM-DENTAL"):
    !git clone https://github.com/rezaxr14/VLM-DENTAL.git

%cd {work_dir}
!git pull

In [ ]:
# Install the project and all its requirements, including SFT dependencies
!pip install -e .
!pip install python-dotenv pandas pillow google-generativeai anthropic huggingface_hub ultralytics
!pip install trl peft bitsandbytes

## 2. Run Supervised Fine-Tuning

In [ ]:
import os

# Set NO_TOOLS = True if training the baseline #3 model (without tools)
NO_TOOLS = False
trace_filename = 'train_cot_traces_no_tools.jsonl' if NO_TOOLS else 'train_cot_traces.jsonl'
dataset_path = f'{work_dir}/data/traces/{trace_filename}'
model_tag = 'qwen3_5_9b_sft_no_tools' if NO_TOOLS else 'qwen3_5_9b_sft'
output_dir = f'{work_dir}/data/models/{model_tag}'

# Automatically fetch verified traces from Hugging Face Hub if not present locally
if not os.path.exists(dataset_path):
    print(f"Trace file {trace_filename} not found locally in {work_dir}/data/traces/.")
    print("Downloading verified traces from Hugging Face Hub (Reza-Nadimi/vlm-dental-traces)...")
    os.system("python scripts/sync_traces_hf.py --download")

!python scripts/train_sft.py \
    --dataset_path {dataset_path} \
    --output_dir {output_dir} \
    --batch_size 1 \
    --epochs 3
